# Meeting Summarizer - Remote Pyannote Diarization

This notebook runs the Pyannote Community-1 diarization pipeline on a Colab T4 GPU and exposes it via an Ngrok HTTP tunnel.

In [ ]:
!pip install pyannote.audio==4.0.0 fastapi uvicorn python-multipart pyngrok nest-asyncio

In [ ]:
import os
import torch
from pyannote.audio import Pipeline

# --- CONFIGURATION ---
# 1. Get your Hugging Face token from https://hf.co/settings/tokens
HF_TOKEN = "your_hugging_face_token_here"

# 2. Get your Ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "your_ngrok_authtoken_here"

# 3. Choose a secret API key to protect your endpoint (must match DIARIZATION_API_KEY in your local .env)
API_KEY = "your_random_secret"

# --- INITIALIZATION ---
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

print("Loading Pyannote Community-1 Pipeline...")
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)
pipeline.to(torch.device("cuda"))
print("Pipeline loaded and moved to GPU.")

In [ ]:
import tempfile
import torchaudio
from fastapi import FastAPI, UploadFile, File, Depends, HTTPException, Header
import uvicorn
from pyngrok import ngrok
import nest_asyncio

app = FastAPI(title="Pyannote Diarization Server")

def verify_api_key(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing Bearer token")
    token = authorization.split(" ")[1]
    if token != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")
    return token

@app.get("/health")
async def health():
    return {
        "status": "ok",
        "cuda": torch.cuda.is_available(),
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    }

@app.post("/diarize")
async def diarize(
    audio: UploadFile = File(...),
    num_speakers: int = None,
    api_key: str = Depends(verify_api_key)
):
    content = await audio.read()
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as tmp:
        tmp.write(content)
        tmp.flush()
        
        # Load waveform into memory using torchaudio
        waveform, sample_rate = torchaudio.load(tmp.name)
        
        kwargs = {}
        if num_speakers is not None:
            kwargs["num_speakers"] = num_speakers
            
        # Run inference on GPU
        output = pipeline({
            "waveform": waveform,
            "sample_rate": sample_rate
        }, **kwargs)
        
        diarization = output.exclusive_speaker_diarization
        
        segments = []
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            segments.append({
                "speaker": speaker,
                "start": float(turn.start),
                "end": float(turn.end)
            })
            
        return {"segments": segments}

# Expose through Ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
public_url = ngrok.connect(8000).public_url
print(f"\n\n====== SERVER IS RUNNING ======")
print(f"Set this in your local .env:")
print(f"DIARIZATION_REMOTE_URL={public_url}")
print(f"===============================\n\n")

# Run FastAPI
nest_asyncio.apply()
uvicorn.run(app, port=8000)